# Narrar lecciones en Kaggle (sin tener que vigilarlo)

Narra las lecciones que falten con la voz de Epachuko y sube cada MP3 a tu repositorio privado de GitHub en cuanto termina.

**Una sola vez:**
1. *Settings* (panel derecho): **Accelerator → GPU T4 x2** e **Internet → On**.
2. *Add-ons → Secrets*: añade **GH_TOKEN** con tu clave de GitHub y marca la casilla para este cuaderno.

**Cada vez:** arriba a la derecha, **Save Version → Save & Run All (Commit)**. Ya puedes cerrar la pestaña: sigue solo hasta terminar (o hasta 11 h) y, si se para, al lanzarlo otra vez sigue por donde iba.

In [ ]:
# 1. Clave de GitHub
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GH_TOKEN")
    print("🔑 Clave de GitHub lista")
except Exception as e:
    raise SystemExit("❌ No encuentro el secreto GH_TOKEN. Añádelo en Add-ons → Secrets y márcalo para este cuaderno.")


In [ ]:
# 2. Instalar el estudio y el motor de voz
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ No hay GPU. En Settings → Accelerator elige GPU T4 x2.")
print("✅", gpu.splitlines()[0])
if os.path.isdir("/kaggle/working/estudio-de-voz/.git"):
    subprocess.run(["git", "-C", "/kaggle/working/estudio-de-voz", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/Jonragitt/estudio-de-voz", "/kaggle/working/estudio-de-voz"], check=True)
r = subprocess.run("pip install -q chatterbox-tts yt-dlp", shell=True, capture_output=True, text=True)
subprocess.run("pip uninstall -y -q torchvision", shell=True)  # el de la imagen rompe transformers con el torch de Chatterbox
r2 = subprocess.run(["python", "-c", "from transformers import LlamaModel; import chatterbox.mtl_tts"], capture_output=True, text=True)
if r2.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:], r2.stderr[-3000:])
    raise SystemExit("❌ El motor de voz no se ha instalado bien (mira el error de arriba).")
print("✅ Motor de voz instalado")


In [ ]:
# 3. Narrar y subir a GitHub
import os, subprocess, sys
env = dict(os.environ,
           REPO="jonragitt/apunty",
           RAMA="claude/project-thread-v133ti",
           TEXTOS="narracion/apunty-japon.json",
           SALIDA="narracion/audio",
           VOZ="narracion/voz")
p = subprocess.Popen([sys.executable, "-u", "kaggle/narrar_repo.py"], cwd="/kaggle/working/estudio-de-voz",
                     env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linea in p.stdout:
    print(linea, end="")
p.wait()
if p.returncode:
    raise SystemExit(f"❌ La narración se ha parado (código {p.returncode}).")
